# Прогнозирование болезней

Действия над числами

In [21]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

In [20]:
keras.utils.set_random_seed(42)

In [22]:
files = glob.glob("**/*.csv", recursive=True)

print("Найденные файлы:", files)

if not files:
    raise FileNotFoundError(
        "CSV-файл не найден. Проверь, что архив heart.zip распакован."
    )

df = pd.read_csv(files[0])

print(df.head())
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())

Найденные файлы: ['heart.csv']
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

   ca        thal  target  
0   0       fixed       0  
1   3      normal       1  
2   2  reversible       0  
3   0      normal       0  
4   0      normal       0  
(303, 14)
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0

In [23]:
# Название столбца, который нужно предсказывать
target_column = "target"

if target_column not in df.columns:
    # Другие распространённые названия целевой переменной
    alternatives = ["HeartDisease", "heart_disease", "output", "num"]
    target_column = next(
        (col for col in alternatives if col in df.columns),
        None
    )

if target_column is None:
    raise ValueError(
        "Не найден целевой столбец. Проверь df.columns.tolist()."
    )

# Удаляем строки с пропущенными значениями
df = df.dropna().copy()

# Признаки и целевая переменная
X = df.drop(columns=[target_column])
y = df[target_column]

# Для бинарной классификации целевая переменная должна содержать 0 и 1
if y.nunique() != 2:
    raise ValueError(
        f"Ожидалось 2 класса, обнаружено: {y.unique()}"
    )

# Если метки не 0 и 1, преобразуем их
if set(y.unique()) != {0, 1}:
    labels = sorted(y.unique())
    y = y.map({labels[0]: 0, labels[1]: 1})

# Преобразуем категориальные признаки в числовые
X = pd.get_dummies(X, drop_first=False, dtype=float)

X = X.astype("float32")
y = y.astype("float32")

print("Количество признаков:", X.shape[1])
print("Количество объектов:", len(X))
print("Распределение классов:")
print(y.value_counts())

Количество признаков: 17
Количество объектов: 303
Распределение классов:
target
0.0    220
1.0     83
Name: count, dtype: int64


In [35]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [34]:
%pip install scikit-learn


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)

Обучающая выборка: (242, 17)
Тестовая выборка: (61, 17)


In [37]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),

    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.2),

    keras.layers.Dense(16, activation="relu"),

    keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

E0000 00:00:1793333349.604640   12948 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.4611 - loss: 0.7250 - val_accuracy: 0.5510 - val_loss: 0.6857
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5596 - loss: 0.6767 - val_accuracy: 0.6122 - val_loss: 0.6665
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6632 - loss: 0.6458 - val_accuracy: 0.6327 - val_loss: 0.6496
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6580 - loss: 0.6325 - val_accuracy: 0.7143 - val_loss: 0.6319
Epoch 5/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7047 - loss: 0.6151 - val_accuracy: 0.7551 - val_loss: 0.6134
Epoch 6/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7617 - loss: 0.5939 - val_accuracy: 0.7755 - val_loss: 0.5934
Epoch 7/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7513 - loss: 0.5708 - val_accuracy: 0.7551 - val_loss: 0.5739
Epoch 8/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7927 - loss: 0.5351 - val_accuracy: 0.7551 - val_loss: 0.553

In [41]:
from sklearn.metrics import classification_report, confusion_matrix

In [42]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")

probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print("\nClassification report:")
print(classification_report(
    y_test.astype(int),
    predictions,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(
    y_test.astype(int),
    predictions
))

Test loss: 0.3816
Test accuracy: 0.8033

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.98      0.88        44
           1       0.86      0.35      0.50        17

    accuracy                           0.80        61
   macro avg       0.83      0.67      0.69        61
weighted avg       0.81      0.80      0.77        61

Confusion matrix:
[[43  1]
 [11  6]]


In [43]:
model.save("heart_model.keras")
print("Модель сохранена.")

Модель сохранена.
